In [1]:
# =========================================
# 06C_signal_boost_features.ipynb
# Build stronger physics-inspired features from Notebook 05 outputs
# =========================================

import os
import json
import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)

WORK_DIR = r"C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026"

INTERIM_DIR = os.path.join(WORK_DIR, "interim")
REPORT_DIR = os.path.join(WORK_DIR, "reports")
META_DIR = os.path.join(WORK_DIR, "metadata")
ARTIFACT_DIR = os.path.join(WORK_DIR, "artifacts", "06C_signal_boost")

for p in [INTERIM_DIR, REPORT_DIR, META_DIR, ARTIFACT_DIR]:
    os.makedirs(p, exist_ok=True)

X_FILE = os.path.join(INTERIM_DIR, "X_features.parquet")
Y_FILE = os.path.join(INTERIM_DIR, "y_target.parquet")
GROUP_FILE = os.path.join(INTERIM_DIR, "groups.parquet")

print("X_FILE:", X_FILE)
print("Y_FILE:", Y_FILE)
print("GROUP_FILE:", GROUP_FILE)

X_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\X_features.parquet
Y_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\y_target.parquet
GROUP_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\groups.parquet


In [2]:
X = pd.read_parquet(X_FILE)
y_df = pd.read_parquet(Y_FILE)
groups_df = pd.read_parquet(GROUP_FILE)

y = y_df["T80_log1p"].copy()
groups = groups_df["Ref_DOI_number"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)
print("Unique DOI groups:", groups.nunique())

X.head()

X shape: (1835, 77)
y shape: (1835,)
groups shape: (1835,)
Unique DOI groups: 964


,is_nip,is_pin,is_other_arch,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,encapsulation_flag,protocol_has_l,protocol_has_d,bias_is_mpp,bias_is_oc,bias_is_sc,light_intensity_suns,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,rh_pct,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition,architecture_family_nip,architecture_family_other_or_unknown,architecture_family_pin,etl_family_c60,etl_family_other_or_unknown,etl_family_other_rare,etl_family_pcbm,etl_family_sno2,etl_family_tio2,etl_family_zno,htl_family_missing,htl_family_niox,htl_family_other_or_unknown,htl_family_other_rare,htl_family_p3ht,htl_family_pedot_pss,htl_family_ptaa,htl_family_spiro_ometad,backcontact_family_ag,backcontact_family_al,backcontact_family_au,backcontact_family_carbon,backcontact_family_cu,backcontact_family_other_rare,protocol_family_isos_d,protocol_family_isos_l,protocol_family_other_isos,protocol_family_other_or_unknown,protocol_family_other_rare,bias_family_mpp,bias_family_open_circuit,bias_family_other_rare,light_bin_dark_or_zero,light_bin_high_light,light_bin_missing,light_bin_other_rare
0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1.59,NaN,NaN,0.060,0.0,0.0,1,0,1,0,0,100.0,0,0,1,25.0,NaN,1,0,0,0,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False
1,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NaN,500.0,25.0,0.160,0.0,0.0,1,0,1,0,0,100.0,0,0,1,25.0,NaN,1,0,0,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False
2,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1.60,350.0,NaN,1.020,0.0,0.0,0,1,0,1,0,0.0,1,0,0,85.0,50.0,0,1,0,1,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False
3,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,1.60,200.0,NaN,0.013,0.0,0.0,1,0,0,1,0,100.0,0,0,1,25.0,40.0,1,0,0,0,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False
4,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,1.96,104.0,NaN,0.040,0.0,0.0,0,1,0,1,0,0.0,1,0,0,25.0,0.0,1,0,1,0,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False


In [3]:
assert len(X) == len(y) == len(groups), "Length mismatch"

print("Missing values in X before cleaning:", int(X.isna().sum().sum()))

X = X.replace([np.inf, -np.inf], np.nan)

num_cols = X.select_dtypes(include=[np.number, bool]).columns.tolist()
X[num_cols] = X[num_cols].apply(pd.to_numeric, errors="coerce")
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

print("Missing values in X after cleaning:", int(X.isna().sum().sum()))
assert X.isna().sum().sum() == 0, "X still contains missing values"

print("Data ready.")

Missing values in X before cleaning: 3974
Missing values in X after cleaning: 0
Data ready.


In [4]:
def find_cols(keywords, cols):
    out = []
    for c in cols:
        cl = c.lower()
        if any(k in cl for k in keywords):
            out.append(c)
    return out

all_cols = X.columns.tolist()

arch_cols = find_cols(["is_nip", "is_pin", "architecture"], all_cols)
etl_cols = find_cols(["etl_"], all_cols)
htl_cols = find_cols(["htl_"], all_cols)
back_cols = find_cols(["back_"], all_cols)
bias_cols = find_cols(["bias_"], all_cols)
light_cols = find_cols(["light"], all_cols)
bandgap_cols = find_cols(["band_gap"], all_cols)
thickness_cols = find_cols(["thickness"], all_cols)
area_cols = find_cols(["area"], all_cols)
additive_cols = find_cols(["additive"], all_cols)

print("Architecture cols:", arch_cols)
print("ETL cols:", etl_cols[:10], " ... n=", len(etl_cols))
print("HTL cols:", htl_cols[:10], " ... n=", len(htl_cols))
print("Bias cols:", bias_cols)
print("Light cols:", light_cols)
print("Bandgap cols:", bandgap_cols)
print("Thickness cols:", thickness_cols)
print("Area cols:", area_cols)
print("Additive cols:", additive_cols)

Architecture cols: ['is_nip', 'is_pin', 'architecture_family_nip', 'architecture_family_other_or_unknown', 'architecture_family_pin']
ETL cols: ['etl_has_tio2', 'etl_has_sno2', 'etl_has_pcbm', 'etl_has_c60', 'etl_has_zno', 'has_etl_additives', 'etl_thickness_nm', 'etl_family_c60', 'etl_family_other_or_unknown', 'etl_family_other_rare']  ... n= 14
HTL cols: ['htl_has_spiro', 'htl_has_ptaa', 'htl_has_pedot', 'htl_has_niox', 'htl_has_p3ht', 'has_htl_additives', 'htl_family_missing', 'htl_family_niox', 'htl_family_other_or_unknown', 'htl_family_other_rare']  ... n= 14
Bias cols: ['bias_is_mpp', 'bias_is_oc', 'bias_is_sc', 'bias_family_mpp', 'bias_family_open_circuit', 'bias_family_other_rare']
Light cols: ['light_intensity_suns', 'is_high_light', 'light_bin_dark_or_zero', 'light_bin_high_light', 'light_bin_missing', 'light_bin_other_rare']
Bandgap cols: ['band_gap_ev']
Thickness cols: ['perovskite_thickness_nm', 'etl_thickness_nm']
Area cols: ['cell_area_measured_cm2']
Additive cols: ['has

In [5]:
X_eng = X.copy()

print("Starting feature count:", X_eng.shape[1])

Starting feature count: 77


In [6]:
# Architecture balance
if "is_nip" in X_eng.columns and "is_pin" in X_eng.columns:
    X_eng["arch_balance"] = X_eng["is_nip"] - X_eng["is_pin"]
    X_eng["arch_sum"] = X_eng["is_nip"] + X_eng["is_pin"]

# Bandgap transforms
for c in bandgap_cols:
    X_eng[f"{c}_sq"] = X_eng[c] ** 2

# Thickness transforms
for c in thickness_cols:
    X_eng[f"{c}_log1p"] = np.log1p(np.clip(X_eng[c], a_min=0, a_max=None))

# Area transforms
for c in area_cols:
    X_eng[f"{c}_log1p"] = np.log1p(np.clip(X_eng[c], a_min=0, a_max=None))

print("After scalar engineering:", X_eng.shape[1])

After scalar engineering: 83


In [7]:
if len(etl_cols) > 0:
    X_eng["etl_signal_sum"] = X_eng[etl_cols].sum(axis=1)
if len(htl_cols) > 0:
    X_eng["htl_signal_sum"] = X_eng[htl_cols].sum(axis=1)
if len(back_cols) > 0:
    X_eng["back_signal_sum"] = X_eng[back_cols].sum(axis=1)
if len(additive_cols) > 0:
    X_eng["additive_signal_sum"] = X_eng[additive_cols].sum(axis=1)
if len(bias_cols) > 0:
    X_eng["bias_signal_sum"] = X_eng[bias_cols].sum(axis=1)
if len(light_cols) > 0:
    X_eng["light_signal_sum"] = X_eng[light_cols].sum(axis=1)

print("After aggregation features:", X_eng.shape[1])

After aggregation features: 89


In [8]:
created_interactions = []

def add_interaction_if_exists(c1, c2, out_name):
    global X_eng
    if c1 in X_eng.columns and c2 in X_eng.columns:
        X_eng[out_name] = X_eng[c1] * X_eng[c2]
        created_interactions.append(out_name)

# Architecture × ETL / HTL
for a in ["is_nip", "is_pin"]:
    for e in ["etl_has_tio2", "etl_has_sno2", "etl_has_pcbm", "etl_has_c60", "etl_has_zno"]:
        add_interaction_if_exists(a, e, f"{a}_x_{e}")
    for h in ["htl_has_spiro", "htl_has_ptaa", "htl_has_pedot", "htl_has_niox", "htl_has_p3ht"]:
        add_interaction_if_exists(a, h, f"{a}_x_{h}")

# Bias / light × stack
for b in ["bias_is_mpp", "bias_is_oc", "bias_is_sc"]:
    for e in ["etl_has_tio2", "etl_has_sno2", "etl_has_pcbm", "etl_has_c60"]:
        add_interaction_if_exists(b, e, f"{b}_x_{e}")
    for h in ["htl_has_spiro", "htl_has_ptaa", "htl_has_pedot", "htl_has_niox"]:
        add_interaction_if_exists(b, h, f"{b}_x_{h}")

for l in ["is_approx_1sun", "is_high_light", "light_intensity_missing"]:
    for e in ["etl_has_tio2", "etl_has_sno2", "etl_has_pcbm", "etl_has_c60"]:
        add_interaction_if_exists(l, e, f"{l}_x_{e}")
    for h in ["htl_has_spiro", "htl_has_ptaa", "htl_has_pedot", "htl_has_niox"]:
        add_interaction_if_exists(l, h, f"{l}_x_{h}")

# Numeric physical × architecture
for n in ["band_gap_ev", "perovskite_thickness_nm", "etl_thickness_nm", "cell_area_measured_cm2"]:
    for a in ["is_nip", "is_pin"]:
        add_interaction_if_exists(n, a, f"{n}_x_{a}")

print("Created interactions:", len(created_interactions))
print("After interaction engineering:", X_eng.shape[1])

Created interactions: 68
After interaction engineering: 157


In [9]:
selector = VarianceThreshold(threshold=0.0)
X_var = pd.DataFrame(
    selector.fit_transform(X_eng),
    columns=X_eng.columns[selector.get_support()],
    index=X_eng.index
)

dropped_low_variance = [c for c in X_eng.columns if c not in X_var.columns]

print("Before variance filter:", X_eng.shape)
print("After variance filter:", X_var.shape)
print("Dropped low-variance:", len(dropped_low_variance))

Before variance filter: (1835, 157)
After variance filter: (1835, 147)
Dropped low-variance: 10


In [10]:
corr = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]

X_final = X_var.drop(columns=to_drop_corr)

print("Before correlation filter:", X_var.shape)
print("After correlation filter:", X_final.shape)
print("Dropped high-correlation:", len(to_drop_corr))

Before correlation filter: (1835, 147)
After correlation filter: (1835, 105)
Dropped high-correlation: 42


In [11]:
print("Final X shape:", X_final.shape)
print("Missing in X_final:", int(X_final.isna().sum().sum()))
print("y shape:", y.shape)
print("groups shape:", groups.shape)

assert len(X_final) == len(y) == len(groups)
assert X_final.isna().sum().sum() == 0

Final X shape: (1835, 105)
Missing in X_final: 0
y shape: (1835,)
groups shape: (1835,)


In [12]:
X_OUT = os.path.join(INTERIM_DIR, "06C_X_features.parquet")
Y_OUT = os.path.join(INTERIM_DIR, "06C_y_target.parquet")
G_OUT = os.path.join(INTERIM_DIR, "06C_groups.parquet")

X_final.to_parquet(X_OUT, index=False)
pd.DataFrame({"T80_log1p": y}).to_parquet(Y_OUT, index=False)
pd.DataFrame({"Ref_DOI_number": groups}).to_parquet(G_OUT, index=False)

print("Saved:", X_OUT)
print("Saved:", Y_OUT)
print("Saved:", G_OUT)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\06C_X_features.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\06C_y_target.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\06C_groups.parquet


In [13]:
feature_report = pd.DataFrame({
    "feature": X_final.columns,
    "dtype": X_final.dtypes.astype(str).values,
    "n_unique": [X_final[c].nunique(dropna=True) for c in X_final.columns]
}).sort_values("feature")

feature_report_path = os.path.join(REPORT_DIR, "06C_feature_report.csv")
feature_report.to_csv(feature_report_path, index=False)

print("Saved:", feature_report_path)
feature_report.head(30)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\reports\06C_feature_report.csv


,feature,dtype,n_unique
58,additive_signal_sum,object,4
13,back_has_ag,object,2
14,back_has_al,object,2
12,back_has_au,object,2
15,back_has_carbon,object,2
57,back_signal_sum,object,3
45,backcontact_family_cu,object,2
46,backcontact_family_other_rare,object,2
19,band_gap_ev,object,73
50,bias_family_other_rare,object,2


In [14]:
card = {
    "notebook": "06C_signal_boost_features.ipynb",
    "input_files": {
        "X_features": X_FILE,
        "y_target": Y_FILE,
        "groups": GROUP_FILE
    },
    "output_files": {
        "X_features": X_OUT,
        "y_target": Y_OUT,
        "groups": G_OUT
    },
    "input_shape": {
        "rows": int(X.shape[0]),
        "cols": int(X.shape[1])
    },
    "output_shape": {
        "rows": int(X_final.shape[0]),
        "cols": int(X_final.shape[1])
    },
    "engineered_features_added": int(X_eng.shape[1] - X.shape[1]),
    "low_variance_dropped": len(dropped_low_variance),
    "high_corr_dropped": len(to_drop_corr)
}

card_path = os.path.join(META_DIR, "06C_signal_boost_card.json")
with open(card_path, "w", encoding="utf-8") as f:
    json.dump(card, f, indent=4)

print("Saved:", card_path)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\metadata\06C_signal_boost_card.json


In [15]:
print("========== 06C SUMMARY ==========")
print("Original X shape:", X.shape)
print("Engineered X shape:", X_eng.shape)
print("Final X shape:", X_final.shape)
print("Engineered features added:", X_eng.shape[1] - X.shape[1])
print("Dropped low variance:", len(dropped_low_variance))
print("Dropped high correlation:", len(to_drop_corr))
print("Final groups:", groups.nunique())
print("Saved 06C artifacts successfully.")

========== 06C SUMMARY ==========
Original X shape: (1835, 77)
Engineered X shape: (1835, 157)
Final X shape: (1835, 105)
Engineered features added: 80
Dropped low variance: 10
Dropped high correlation: 42
Final groups: 964
Saved 06C artifacts successfully.
